# Notebook 02: Data Cleaning and Preparation
**Mục tiêu:** 
1. Nhập dữ liệu thô từ FRED.
2. Xử lý các giá trị khuyết thiếu (NaN) và các ngày nghỉ lễ trên thị trường chứng khoán.
3. Đồng bộ hóa tần suất (Resampling) của cả 3 chuỗi dữ liệu (NASDAQ, CPI, FFR) về tần suất cuối tháng.
4. Gộp (Merge) dữ liệu thành một Master DataFrame duy nhất và xuất file.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình hiển thị
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="talk")

# Thiết lập đường dẫn thư mục bằng pathlib
ROOT = Path.cwd().resolve().parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

# Đảm bảo thư mục processed tồn tại
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

RAW_DIR: S:\vscode\nasdaq-macro-shocks-analysis\data\raw
PROCESSED_DIR: S:\vscode\nasdaq-macro-shocks-analysis\data\processed


## 1. Load Raw Data
Dữ liệu tải từ FRED thường có một đặc điểm: các ngày không có dữ liệu (hoặc lỗi) thường được điền bằng ký tự `.` (dấu chấm). Chúng ta sẽ yêu cầu Pandas nhận diện ký tự này là `NaN` ngay lúc đọc file.

In [2]:
# Đọc file CSV, ép các giá trị '.' thành NaN
# Đảm bảo bạn đã tải 3 file này từ FRED và để đúng tên trong thư mục data/raw/
ndx_raw = pd.read_csv(RAW_DIR / "NASDAQ100.csv", na_values='.')
cpi_raw = pd.read_csv(RAW_DIR / "CPIAUCSL.csv", na_values='.')
ffr_raw = pd.read_csv(RAW_DIR / "FEDFUNDS.csv", na_values='.')

print(f"Bản ghi NASDAQ ban đầu: {len(ndx_raw)}")
print(f"Bản ghi CPI ban đầu: {len(cpi_raw)}")
print(f"Bản ghi FFR ban đầu: {len(ffr_raw)}")

Bản ghi NASDAQ ban đầu: 10530
Bản ghi CPI ban đầu: 952
Bản ghi FFR ban đầu: 862


## 2. Cleaning and Resampling to Monthly Frequency
Thị trường chứng khoán đóng cửa vào cuối tuần và ngày lễ, tạo ra các "khoảng trống" (gaps) trong chuỗi thời gian. Trong khi đó, CPI và FFR lại được báo cáo theo tháng. 

**Chiến lược:**
1. Chuyển cột `DATE` thành DatetimeIndex.
2. **Đối với NASDAQ:** Dùng `ffill()` (Forward Fill) để mượn giá đóng cửa của ngày hôm trước điền cho ngày nghỉ lễ. Sau đó, hạ tần suất (downsample) xuống ngày cuối cùng của tháng (`ME` - Month End).
3. **Đối với CPI và FFR:** Các chỉ số này báo cáo vào ngày đầu tháng, ta sẽ dời mốc thời gian về ngày cuối tháng (`ME`) để đồng bộ hoàn hảo với giá đóng cửa của chứng khoán.

In [3]:
def process_time_series(df, value_col, resample_method='last'):
    """
    Hàm chuẩn hóa DatetimeIndex và hạ tần suất (resample) về cuối tháng (Month End).
    Lưu ý: Kể từ Pandas 2.2.0, bí danh 'M' đã bị deprecated, sử dụng 'ME' (Month End).
    """
    df = df.copy()
    
    # 1. Ép kiểu về datetime và set index
    df['DATE'] = pd.to_datetime(df['DATE'])
    df.set_index('DATE', inplace=True)
    
    # 2. Đảm bảo cột giá trị là dạng số (numeric)
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    
    # 3. Forward fill để lấp đầy các ngày nghỉ lễ/cuối tuần (chỉ tác dụng rõ với data ngày như NDX)
    df = df.ffill()
    
    # 4. Resample về ngày cuối cùng của tháng
    if resample_method == 'last':
        df_monthly = df.resample('ME').last()
    elif resample_method == 'mean':
        df_monthly = df.resample('ME').mean()
        
    return df_monthly

# Thực thi quá trình
# Lấy giá đóng cửa ngày cuối tháng cho NASDAQ
# Ensure raw dataframes use 'DATE' as the datetime column name
for df in (ndx_raw, cpi_raw, ffr_raw):
    if 'observation_date' in df.columns and 'DATE' not in df.columns:
        df.rename(columns={'observation_date': 'DATE'}, inplace=True)

ndx_monthly = process_time_series(ndx_raw, 'NASDAQ100', resample_method='last')

# CPI phản ánh mức giá tại một thời điểm, lấy giá trị cuối cùng
cpi_monthly = process_time_series(cpi_raw, 'CPIAUCSL', resample_method='last')

# FFR (FEDFUNDS) vốn dĩ đã là tỷ lệ trung bình tháng của FRED, ta align nó về ngày cuối tháng
ffr_monthly = process_time_series(ffr_raw, 'FEDFUNDS', resample_method='last')

print("Hoàn tất Resampling!")
display(ndx_monthly.head(3))

Hoàn tất Resampling!


,NASDAQ100
DATE,
1986-01-31,132.93
1986-02-28,140.43
1986-03-31,148.86


## 3. Merge into Master DataFrame
Sử dụng `join` với phương thức `inner` để kết hợp 3 chuỗi dữ liệu. Điều này sẽ tự động loại bỏ những khoảng thời gian lệch nhau (ví dụ: dữ liệu FFR có từ 1954, nhưng NASDAQ chỉ có từ 1985, `inner join` sẽ chỉ giữ lại dải thời gian từ 1985 trở đi).

In [4]:
# Gộp 3 dataframe dựa trên DatetimeIndex
master_df = ndx_monthly.join([cpi_monthly, ffr_monthly], how='inner')

# Drop bất kỳ hàng nào còn sót NaN (để cẩn thận)
master_df.dropna(inplace=True)

# Đổi tên cột cho chuẩn mực và dễ gõ code sau này
master_df.rename(columns={
    'NASDAQ100': 'NASDAQ',
    'CPIAUCSL': 'CPI',
    'FEDFUNDS': 'FFR'
}, inplace=True)

print(f"Dải thời gian chung (Intersection): {master_df.index.min().date()} đến {master_df.index.max().date()}")
print(f"Tổng số tháng quan sát: {len(master_df)}")

display(master_df.head())
display(master_df.tail())

Dải thời gian chung (Intersection): 1986-01-31 đến 2026-04-30
Tổng số tháng quan sát: 484


,NASDAQ,CPI,FFR
DATE,,,
1986-01-31,132.93,109.9,8.14
1986-02-28,140.43,109.7,7.86
1986-03-31,148.86,109.1,7.48
1986-04-30,154.91,108.7,6.99
1986-05-31,163.16,109.0,6.85


,NASDAQ,CPI,FFR
DATE,,,
2025-12-31,25249.85,326.031,3.72
2026-01-31,25552.39,326.588,3.64
2026-02-28,24960.04,327.460,3.64
2026-03-31,23740.19,330.293,3.64
2026-04-30,27452.12,332.407,3.64


## 4. Final Validation & Export
Trước khi lưu, chúng ta kiểm tra nhanh một lần cuối xem có còn giá trị `NaN` nào lọt qua không, và kiểm tra tính liên tục của chuỗi tháng (đảm bảo không bị nhảy cóc tháng nào).

In [5]:
# 1. Kiểm tra Missing Values
missing_counts = master_df.isna().sum()
assert missing_counts.sum() == 0, "CẢNH BÁO: Dữ liệu vẫn còn chứa giá trị NaN!"

# 2. Kiểm tra tính liên tục của thời gian (Tùy chọn nhưng Rất tốt cho Time Series)
# Tạo một dải tháng liên tục từ min đến max của index thực tế
expected_index = pd.date_range(start=master_df.index.min(), end=master_df.index.max(), freq='ME')
missing_months = expected_index.difference(master_df.index)

if len(missing_months) == 0:
    print("✓ Chuỗi thời gian liên tục hoàn hảo, không bị đứt gãy tháng nào.")
else:
    print(f"⚠ PHÁT HIỆN LỖI: Bị khuyết {len(missing_months)} tháng trong dữ liệu!")
    print(missing_months)

# 3. Xuất file Master Data
output_path = PROCESSED_DIR / "master_data_monthly.csv"
master_df.to_csv(output_path)

print(f"\nĐã xuất thành công dữ liệu sạch ra: {output_path}")

✓ Chuỗi thời gian liên tục hoàn hảo, không bị đứt gãy tháng nào.

Đã xuất thành công dữ liệu sạch ra: S:\vscode\nasdaq-macro-shocks-analysis\data\processed\master_data_monthly.csv
